In [ ]:

!pip install pyspark pandas matplotlib


In [ ]:

from google.colab import files

uploaded = files.upload()


In [ ]:

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LAKE2026_Rainfall_Analytics") \
    .master("local[*]") \
    .getOrCreate()

print("PySpark started successfully!")
print("Spark version:", spark.version)


In [ ]:

import os

csv_files = [f for f in os.listdir("/content") if f.endswith(".csv")]

print("Available CSV files:")
for f in csv_files:
    print(f)


In [ ]:

# Load both datasets
state_df = spark.read.csv(
    "/content/" + csv_files[0],
    header=True,
    inferSchema=True
)

district_df = spark.read.csv(
    "/content/" + csv_files[1],
    header=True,
    inferSchema=True
)

print("State dataset rows:", state_df.count())
print("District dataset rows:", district_df.count())

print("\nState Dataset:")
state_df.show(5, truncate=False)

print("\nDistrict Dataset:")
district_df.show(5, truncate=False)


In [ ]:
state_df.printSchema()
district_df.printSchema()

In [ ]:

from pyspark.sql import SparkSession

# Load state-level rainfall data
state_df = spark.read.csv(
    "/content/acd3023c-df96-4af8-8815-a7393cb80839.csv",
    header=True,
    inferSchema=True
)

# Load district-level rainfall data
district_df = spark.read.csv(
    "/content/8752174f-1d17-4aaf-8058-2eb396f50157.csv",
    header=True,
    inferSchema=True
)

# Display record counts
print("STATE DATASET")
print("Total records:", state_df.count())

print("\nDISTRICT DATASET")
print("Total records:", district_df.count())


In [ ]:

print("STATE DATASET SCHEMA")
state_df.printSchema()

print("\nDISTRICT DATASET SCHEMA")
district_df.printSchema()

print("\nSTATE DATA SAMPLE")
state_df.show(5, truncate=False)

print("\nDISTRICT DATA SAMPLE")
district_df.show(5, truncate=False)


In [ ]:
state_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .csv("/content/acd3023c-df96-4af8-8815-a7393cb80839.csv")

district_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .csv("/content/8752174f-1d17-4aaf-8058-2eb396f50157.csv")

print("STATE RECORDS:", state_df.count())
print("DISTRICT RECORDS:", district_df.count())

print("\nSTATE COLUMNS:")
print(state_df.columns)

print("\nDISTRICT COLUMNS:")
print(district_df.columns)

print("\nSTATE SAMPLE:")
state_df.show(5, truncate=False)

print("\nDISTRICT SAMPLE:")
district_df.show(5, truncate=False)

In [ ]:
from pyspark.sql.functions import col, to_date

# Clean column names
def clean_columns(df):
    new_names = [
        "_".join(name.strip().replace("\n", " ").replace("\r", "").split())
        for name in df.columns
    ]
    return df.toDF(*new_names)

state_df = clean_columns(state_df)
district_df = clean_columns(district_df)

# Convert date and rainfall columns
for df_name in ["state_df", "district_df"]:
    df = globals()[df_name]

    df = df.withColumn(
        "Date",
        to_date(col("Date"), "yyyy-MM-dd")
    )

    df = df.withColumn(
        "Daily_Actual",
        col("Daily_Actual").cast("double")
    )

    df = df.withColumn(
        "Daily_Normal",
        col("Daily_Normal").cast("double")
    )

    globals()[df_name] = df

# Display cleaned columns
print("STATE COLUMNS:")
print(state_df.columns)

print("\nDISTRICT COLUMNS:")
print(district_df.columns)

print("\nSTATE DATA TYPES:")
state_df.printSchema()

print("\nDISTRICT DATA TYPES:")
district_df.printSchema()

In [ ]:
from pyspark.sql.functions import col, round as spark_round

# Filter India national-level records
india_df = state_df.filter(
    col("State") == "COUNTRY : INDIA"
)

# Keep the required date range
india_df = india_df.filter(
    (col("Date") >= "2026-08-19") &
    (col("Date") <= "2026-10-21")
)

# Calculate daily rainfall departure percentage
india_df = india_df.withColumn(
    "Departure_Percent",
    spark_round(
        (
            (col("Daily_Actual") - col("Daily_Normal"))
            / col("Daily_Normal")
        ) * 100,
        2
    )
)

# Sort records by date
india_df = india_df.orderBy("Date")

print("INDIA NATIONAL RAINFALL DATA")
india_df.select(
    "Date",
    "Daily_Actual",
    "Daily_Normal",
    "Departure_Percent"
).show(15, truncate=False)

print("Total India records:", india_df.count())

In [ ]:
from pyspark.sql.functions import sum as spark_sum, round as spark_round

india_summary = india_df.agg(
    spark_round(spark_sum("Daily_Actual"), 2).alias("Total_Actual_Rainfall"),
    spark_round(spark_sum("Daily_Normal"), 2).alias("Total_Normal_Rainfall")
)

india_summary = india_summary.withColumn(
    "Overall_Departure_Percent",
    spark_round(
        (
            (col("Total_Actual_Rainfall") - col("Total_Normal_Rainfall"))
            / col("Total_Normal_Rainfall")
        ) * 100,
        2
    )
)

india_summary.show(truncate=False)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Convert the small national dataset to Pandas for plotting
india_pd = india_df.select(
    "Date",
    "Daily_Actual",
    "Daily_Normal"
).toPandas()

# Sort by date
india_pd = india_pd.sort_values("Date")

# Create graph
plt.figure(figsize=(14, 6))

plt.plot(
    india_pd["Date"],
    india_pd["Daily_Actual"],
    label="Actual Rainfall",
    marker="o",
    linewidth=2
)

plt.plot(
    india_pd["Date"],
    india_pd["Daily_Normal"],
    label="Normal Rainfall",
    marker="o",
    linewidth=2
)

plt.title(
    "India: Daily Actual vs Normal Rainfall\n"
    "(19 August – 21 October 2026)",
    fontsize=15
)

plt.xlabel("Date", fontsize=12)
plt.ylabel("Rainfall (mm)", fontsize=12)

plt.xticks(rotation=45)
plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=5))
plt.gca().xaxis.set_major_formatter(
    mdates.DateFormatter("%d %b")
)

plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

# Save high-resolution graph
plt.savefig(
    "india_actual_vs_normal_rainfall.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from google.colab import files

files.download("india_actual_vs_normal_rainfall.png")

In [ ]:
from pyspark.sql.functions import (
    col, sum as spark_sum, round as spark_round
)

# Remove national and regional aggregate records
state_analysis = state_df.filter(
    (~col("State").startswith("COUNTRY")) &
    (~col("State").startswith("REGION"))
)

# Calculate state-wise rainfall totals
state_summary = state_analysis.groupBy("State").agg(
    spark_sum("Daily_Actual").alias("Total_Actual"),
    spark_sum("Daily_Normal").alias("Total_Normal")
)

# Calculate rainfall departure percentage
state_summary = state_summary.withColumn(
    "Departure_Percent",
    spark_round(
        (
            (col("Total_Actual") - col("Total_Normal"))
            / col("Total_Normal")
        ) * 100,
        2
    )
)

# Sort states from largest deficit to largest surplus
state_summary = state_summary.orderBy("Departure_Percent")

print("STATES WITH HIGHEST RAINFALL DEFICIT")
state_summary.show(10, truncate=False)

print("STATES WITH HIGHEST RAINFALL SURPLUS")
state_summary.orderBy(
    col("Departure_Percent").desc()
).show(10, truncate=False)

print("Total states/UTs analysed:", state_summary.count())

In [ ]:
from pyspark.sql.functions import (
    col, sum as spark_sum, round as spark_round, countDistinct
)

# Filter the required study period and exclude aggregate rows
state_analysis = state_df.filter(
    (col("Date") >= "2026-08-19") &
    (col("Date") <= "2026-10-21") &
    (~col("State").startswith("COUNTRY")) &
    (~col("State").startswith("REGION"))
)

# Calculate state-wise rainfall totals
state_summary = state_analysis.groupBy("State").agg(
    spark_sum("Daily_Actual").alias("Total_Actual"),
    spark_sum("Daily_Normal").alias("Total_Normal"),
    countDistinct("Date").alias("Days_Recorded")
)

# Calculate percentage departure
state_summary = state_summary.withColumn(
    "Departure_Percent",
    spark_round(
        (
            (col("Total_Actual") - col("Total_Normal"))
            / col("Total_Normal")
        ) * 100,
        2
    )
)

# Show largest deficits
print("TOP 10 STATES WITH RAINFALL DEFICIT")

state_summary.orderBy(
    col("Departure_Percent").asc()
).show(10, truncate=False)

# Show largest surpluses
print("TOP 10 STATES WITH RAINFALL SURPLUS")

state_summary.orderBy(
    col("Departure_Percent").desc()
).show(10, truncate=False)

print("Total states/UTs:", state_summary.count())

In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql.functions import col

# Get 10 states with the highest deficits
deficit_pd = state_summary.orderBy(
    col("Departure_Percent").asc()
).limit(10).toPandas()

# Get 10 states with the highest surpluses
surplus_pd = state_summary.orderBy(
    col("Departure_Percent").desc()
).limit(10).toPandas()

# Combine both datasets
plot_pd = __import__("pandas").concat(
    [deficit_pd, surplus_pd]
)

# Sort for plotting
plot_pd = plot_pd.sort_values("Departure_Percent")

# Create chart
plt.figure(figsize=(12, 9))

colors = [
    "red" if value < 0 else "green"
    for value in plot_pd["Departure_Percent"]
]

plt.barh(
    plot_pd["State"],
    plot_pd["Departure_Percent"],
    color=colors
)

plt.axvline(0, color="black", linewidth=1)

plt.title(
    "State-wise Rainfall Departure in India\n"
    "(19 August – 21 October 2026)",
    fontsize=15
)

plt.xlabel("Rainfall Departure (%)", fontsize=12)
plt.ylabel("States / Union Territories", fontsize=12)

plt.grid(axis="x", linestyle="--", alpha=0.4)

# Add percentage labels
for index, value in enumerate(plot_pd["Departure_Percent"]):
    plt.text(
        value + (1 if value >= 0 else -1),
        index,
        f"{value:.2f}%",
        va="center",
        ha="left" if value >= 0 else "right",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    "state_wise_rainfall_departure.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from google.colab import files

files.download("state_wise_rainfall_departure.png")

In [ ]:
from pyspark.sql.functions import (
    col, sum as spark_sum,
    countDistinct, round as spark_round
)

# Filter the study period
district_analysis = district_df.filter(
    (col("Date") >= "2026-08-19") &
    (col("Date") <= "2026-10-21")
)

# Group by state and district
district_summary = district_analysis.groupBy(
    "State", "District"
).agg(
    spark_sum("Daily_Actual").alias("Total_Actual"),
    spark_sum("Daily_Normal").alias("Total_Normal"),
    countDistinct("Date").alias("Days_Recorded")
)

# Keep districts with all 64 daily observations
district_summary = district_summary.filter(
    col("Days_Recorded") == 64
)

# Calculate rainfall departure
district_summary = district_summary.withColumn(
    "Departure_Percent",
    spark_round(
        (
            (col("Total_Actual") - col("Total_Normal"))
            / col("Total_Normal")
        ) * 100,
        2
    )
)

# Show 10 districts with highest deficit
print("TOP 10 DISTRICTS WITH RAINFALL DEFICIT")

district_summary.orderBy(
    col("Departure_Percent").asc()
).show(10, truncate=False)

# Show 10 districts with highest surplus
print("TOP 10 DISTRICTS WITH RAINFALL SURPLUS")

district_summary.orderBy(
    col("Departure_Percent").desc()
).show(10, truncate=False)

print("Total districts analysed:", district_summary.count())

In [ ]:
# Inspect districts with zero total actual rainfall
zero_districts = district_summary.filter(
    col("Total_Actual") == 0
)

print("DISTRICTS WITH ZERO TOTAL ACTUAL RAINFALL:")
zero_districts.select(
    "State",
    "District",
    "Total_Actual",
    "Total_Normal",
    "Days_Recorded"
).show(20, truncate=False)

# Inspect daily records for Lower Dibang Valley
print("\nLOWER DIBANG VALLEY DAILY RECORDS:")

district_analysis.filter(
    col("District") == "LOWER DIBANG VALLEY"
).select(
    "Date",
    "Daily_Actual",
    "Daily_Normal"
).orderBy("Date").show(20, truncate=False)

In [ ]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum, when,
    min as spark_min, max as spark_max
)

# Check all four districts
districts_to_check = [
    "LOWER DIBANG VALLEY",
    "DIBANG VALLEY",
    "NORTH DELHI",
    "KARGIL"
]

for district in districts_to_check:

    print("\nDISTRICT:", district)

    district_analysis.filter(
        col("District") == district
    ).agg(
        count("*").alias("Total_Records"),
        spark_sum(
            when(col("Daily_Actual") == 0, 1).otherwise(0)
        ).alias("Zero_Rainfall_Days"),
        spark_sum(
            when(col("Daily_Actual").isNull(), 1).otherwise(0)
        ).alias("Missing_Rainfall_Days"),
        spark_min("Daily_Actual").alias("Minimum_Actual"),
        spark_max("Daily_Actual").alias("Maximum_Actual")
    ).show()

In [ ]:

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql.functions import col

# -----------------------------------------
# 1. PREPARE DISTRICT DATA
# -----------------------------------------

# Exclude districts with zero actual rainfall
valid_districts = district_summary.filter(
    (col("Total_Actual") > 0) &
    (col("Total_Normal") > 0)
)

# Top 10 rainfall deficit districts
deficit_pd = valid_districts.orderBy(
    col("Departure_Percent").asc()
).limit(10).toPandas()

# Top 10 rainfall surplus districts
surplus_pd = valid_districts.orderBy(
    col("Departure_Percent").desc()
).limit(10).toPandas()

# Combine datasets
plot_pd = pd.concat(
    [deficit_pd, surplus_pd],
    ignore_index=True
)

# Create district labels
plot_pd["Label"] = (
    plot_pd["District"].str.title()
    + " (" +
    plot_pd["State"].str.title()
    + ")"
)

# Sort values
plot_pd = plot_pd.sort_values(
    "Departure_Percent",
    ascending=True
).reset_index(drop=True)


# -----------------------------------------
# 2. CREATE POSTER-QUALITY GRAPH
# -----------------------------------------

fig, ax = plt.subplots(figsize=(16, 12))

# Colors for rainfall deficit and surplus
colors = [
    "#D62828" if value < 0 else "#16803C"
    for value in plot_pd["Departure_Percent"]
]

# Horizontal bar chart
bars = ax.barh(
    plot_pd["Label"],
    plot_pd["Departure_Percent"],
    color=colors,
    edgecolor="white",
    linewidth=1.5,
    height=0.72
)


# -----------------------------------------
# 3. TITLE AND AXIS FORMATTING
# -----------------------------------------

ax.set_title(
    "DISTRICT-WISE RAINFALL VARIABILITY IN INDIA\n"
    "19 AUGUST – 21 OCTOBER 2026",
    fontsize=21,
    fontweight="bold",
    pad=24
)

ax.set_xlabel(
    "RAINFALL DEPARTURE (%)",
    fontsize=15,
    fontweight="bold",
    labelpad=12
)

ax.set_ylabel(
    "DISTRICT (STATE)",
    fontsize=15,
    fontweight="bold",
    labelpad=12
)

# Bold district labels
ax.tick_params(axis="y", labelsize=11)
ax.tick_params(axis="x", labelsize=11)

for label in ax.get_yticklabels():
    label.set_fontweight("bold")

for label in ax.get_xticklabels():
    label.set_fontweight("bold")


# -----------------------------------------
# 4. ZERO REFERENCE LINE AND GRID
# -----------------------------------------

ax.axvline(
    0,
    color="#222222",
    linewidth=2,
    linestyle="-"
)

ax.grid(
    axis="x",
    linestyle="--",
    linewidth=0.8,
    alpha=0.3
)

ax.set_axisbelow(True)

# Give labels enough room
minimum = plot_pd["Departure_Percent"].min()
maximum = plot_pd["Departure_Percent"].max()

ax.set_xlim(
    minimum - 18,
    maximum + 25
)


# -----------------------------------------
# 5. ADD BOLD PERCENTAGE LABELS
# -----------------------------------------

for bar, value in zip(
    bars,
    plot_pd["Departure_Percent"]
):

    y_position = bar.get_y() + bar.get_height() / 2

    if value < 0:

        # Place deficit labels inside red bars
        ax.text(
            value + 3,
            y_position,
            f"{value:.2f}%",
            va="center",
            ha="left",
            fontsize=10,
            fontweight="bold",
            color="white"
        )

    else:

        # Place surplus labels outside green bars
        ax.text(
            value + 3,
            y_position,
            f"+{value:.2f}%",
            va="center",
            ha="left",
            fontsize=10,
            fontweight="bold",
            color="#14532D"
        )


# -----------------------------------------
# 6. LEGEND
# -----------------------------------------

from matplotlib.patches import Patch

legend_items = [
    Patch(
        facecolor="#D62828",
        label="Below Normal Rainfall (Deficit)"
    ),
    Patch(
        facecolor="#16803C",
        label="Above Normal Rainfall (Surplus)"
    )
]

ax.legend(
    handles=legend_items,
    loc="lower right",
    fontsize=11,
    frameon=True,
    edgecolor="gray"
)


# -----------------------------------------
# 7. FINAL LAYOUT AND SAVE
# -----------------------------------------

fig.patch.set_facecolor("white")
ax.set_facecolor("#FAFAFA")

plt.tight_layout()

plt.savefig(
    "district_wise_rainfall_variability_poster.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()


In [ ]:
from google.colab import files

files.download("district_wise_rainfall_variability.png")

In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:

# STEP 2: Install Java and PySpark

# Install Java 17
!apt-get update -qq
!apt-get install openjdk-17-jdk-headless -qq

# Install PySpark
!pip install -q pyspark

# Configure Java
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] += ":" + os.environ["JAVA_HOME"] + "/bin"

# Import PySpark
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("LAKE2026_Rainfall_Analysis") \
    .master("local[*]") \
    .getOrCreate()

print("Java and PySpark configured successfully!")
print("Spark Version:", spark.version)


In [ ]:

# STEP 3: Load rainfall dataset into PySpark

file_path = "/content/rainfall_2024_2025_combined (1).csv.gz"

# Read compressed CSV using Spark
rainfall_df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

# Display first 10 records
rainfall_df.show(10)

# Display dataset structure
rainfall_df.printSchema()

# Count total records
print("Total Records:", rainfall_df.count())


In [ ]:

# STEP 4: Data Quality Checks

from pyspark.sql.functions import (
    col, count, when, min, max, sum, isnan
)

# 1. Check missing values in each column
print("MISSING VALUES")

rainfall_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in rainfall_df.columns
]).show()

# 2. Check date range
print("DATE RANGE")

rainfall_df.select(
    min("date").alias("Start Date"),
    max("date").alias("End Date")
).show()

# 3. Check rainfall statistics
print("RAINFALL STATISTICS")

rainfall_df.select(
    min("rainfall_mm").alias("Minimum Rainfall"),
    max("rainfall_mm").alias("Maximum Rainfall"),
    count("rainfall_mm").alias("Valid Rainfall Records")
).show()

# 4. Check negative rainfall values
print("NEGATIVE RAINFALL VALUES")

rainfall_df.filter(
    col("rainfall_mm") < 0
).count()

# 5. Check duplicate records
print("DUPLICATE RECORDS")

duplicate_count = (
    rainfall_df.groupBy(
        "date", "latitude", "longitude"
    )
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate records:", duplicate_count)


In [ ]:
from pyspark.sql import functions as F

# STEP 1: Count total records
total_records = rainfall_df.count()

# STEP 2: Count missing values in each column
missing_values = rainfall_df.select([
    F.count(
        F.when(F.col(column).isNull(), column)
    ).alias(column)
    for column in rainfall_df.columns
])

print("Missing Values in Each Column:")
missing_values.show()

# STEP 3: Count missing rainfall values
missing_rainfall = rainfall_df.filter(
    F.col("rainfall_mm").isNull()
).count()

# STEP 4: Count valid rainfall records
valid_rainfall = rainfall_df.filter(
    F.col("rainfall_mm").isNotNull()
).count()

# STEP 5: Calculate missing rainfall percentage
missing_percentage = (
    missing_rainfall / total_records
) * 100

# STEP 6: Calculate valid rainfall percentage
valid_percentage = (
    valid_rainfall / total_records
) * 100

# STEP 7: Display results
print("\n========== RAINFALL DATA QUALITY REPORT ==========")

print("Total Records:", total_records)

print("Missing Rainfall Records:", missing_rainfall)

print("Valid Rainfall Records:", valid_rainfall)

# FIX: Display missing percentage correctly
print(
    "\nMissing rainfall percentage:",
    f"{missing_percentage:.2f}%"
)

print(
    "Valid rainfall percentage:",
    f"{valid_percentage:.2f}%"
)

print("=================================================")

In [ ]:
from pyspark.sql import functions as F

# Filter valid rainfall observations
valid_df = rainfall_df.filter(
    F.col("rainfall_mm").isNotNull()
)

# Calculate rainfall statistics
rainfall_statistics = valid_df.select(
    F.min("rainfall_mm").alias("Minimum Rainfall (mm)"),
    F.max("rainfall_mm").alias("Maximum Rainfall (mm)"),
    F.avg("rainfall_mm").alias("Average Rainfall (mm)"),
    F.stddev("rainfall_mm").alias("Standard Deviation (mm)")
)

# Display results
print("========== RAINFALL STATISTICS ==========")
rainfall_statistics.show(truncate=False)

In [ ]:
# Import required libraries

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import functions as F

# Set graph style
sns.set_theme(style="whitegrid")

# Create output folder
output_folder = "/content/LAKE2026_Rainfall_Graphs"
os.makedirs(output_folder, exist_ok=True)

# Remove missing rainfall observations
valid_df = rainfall_df.filter(
    F.col("rainfall_mm").isNotNull() &
    (~F.isnan("rainfall_mm"))
)

print("Total records:", rainfall_df.count())
print("Valid rainfall records:", valid_df.count())

print("Graph output folder:", output_folder)

In [ ]:
daily_df = valid_df.groupBy("date").agg(
    F.avg("rainfall_mm").alias("daily_avg_rainfall"),
    F.count("rainfall_mm").alias("valid_grid_cells")
).orderBy("date")

daily_pd = daily_df.toPandas()

daily_pd["date"] = pd.to_datetime(daily_pd["date"])
daily_pd["year"] = daily_pd["date"].dt.year
daily_pd["month"] = daily_pd["date"].dt.month

print(daily_pd.head())
print("Number of days with valid rainfall:", len(daily_pd))

In [ ]:
monthly_pd = (
    daily_pd.groupby(["year", "month"])["daily_avg_rainfall"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=monthly_pd,
    x="month",
    y="daily_avg_rainfall",
    hue="year",
    palette={2024: "#1565C0", 2025: "#E65100"},
    marker="o",
    linewidth=2.5,
    markersize=8
)

plt.title("Monthly Average Rainfall Comparison (2024–2025)")
plt.xlabel("Month")
plt.ylabel("Average Daily Spatial Rainfall (mm)")
plt.xticks(range(1, 13))
plt.legend(title="Year")
plt.tight_layout()

plt.savefig(
    f"{output_folder}/01_monthly_rainfall_trend.png",
    dpi=300
)

plt.show()

In [ ]:
annual_pd = (
    daily_pd.groupby("year")["daily_avg_rainfall"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 6))

sns.barplot(
    data=annual_pd,
    x="year",
    y="daily_avg_rainfall"
)

plt.title("Annual Average Daily Spatial Rainfall (2024–2025)")
plt.xlabel("Year")
plt.ylabel("Average Daily Spatial Rainfall (mm)")

plt.tight_layout()

plt.savefig(
    f"{output_folder}/02_annual_comparison.png",
    dpi=300
)

plt.show()

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Pre-Monsoon"
    elif month in [6, 7, 8, 9]:
        return "Southwest Monsoon"
    else:
        return "Post-Monsoon"

daily_pd["season"] = daily_pd["month"].apply(get_season)

seasonal_pd = (
    daily_pd.groupby(["year", "season"])["daily_avg_rainfall"]
    .mean()
    .reset_index()
)

season_order = [
    "Winter",
    "Pre-Monsoon",
    "Southwest Monsoon",
    "Post-Monsoon"
]

plt.figure(figsize=(11, 6))

sns.barplot(
    data=seasonal_pd,
    x="season",
    y="daily_avg_rainfall",
    hue="year",
    order=season_order
)

plt.title("Seasonal Rainfall Comparison (2024–2025)")
plt.xlabel("Season")
plt.ylabel("Average Daily Spatial Rainfall (mm)")
plt.xticks(rotation=15)
plt.tight_layout()

plt.savefig(
    f"{output_folder}/03_seasonal_rainfall.png",
    dpi=300
)

plt.show()

In [ ]:
spatial_df = valid_df.groupBy(
    "latitude",
    "longitude"
).agg(
    F.avg("rainfall_mm").alias("mean_rainfall")
)

spatial_pd = spatial_df.toPandas()

plt.figure(figsize=(12, 8))

scatter = plt.scatter(
    spatial_pd["longitude"],
    spatial_pd["latitude"],
    c=spatial_pd["mean_rainfall"],
    cmap="turbo",
    s=18,
    alpha=0.95,
    edgecolors="none"
)

cbar = plt.colorbar(scatter)
cbar.set_label(
    "Mean Rainfall (mm)",
    fontsize=12,
    fontweight="bold"
)

plt.title(
    "Geographical Distribution of Average Rainfall",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Longitude", fontsize=12, fontweight="bold")
plt.ylabel("Latitude", fontsize=12, fontweight="bold")

plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

plt.tight_layout()

plt.savefig(
    f"{output_folder}/04_spatial_rainfall.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Calculate monthly average rainfall
monthly_heatmap = (
    valid_df
    .groupBy(
        F.year("date").alias("year"),
        F.month("date").alias("month")
    )
    .agg(
        F.avg("rainfall_mm").alias("average_rainfall")
    )
)

# Convert to Pandas
heatmap_pd = monthly_heatmap.toPandas()

# Convert month numbers into names
month_names = {
    1: "Jan", 2: "Feb", 3: "Mar",
    4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sep",
    10: "Oct", 11: "Nov", 12: "Dec"
}

heatmap_pd["month"] = heatmap_pd["month"].map(month_names)

# Arrange data for heatmap
heatmap_data = heatmap_pd.pivot(
    index="year",
    columns="month",
    values="average_rainfall"
)

# Maintain correct month order
heatmap_data = heatmap_data[
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
]

# Create heatmap
plt.figure(figsize=(14, 5))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=1.5,
    linecolor="white",
    cbar_kws={"label": "Average Rainfall (mm)"},
    annot_kws={
        "size": 12,
        "weight": "bold"
    }
)

plt.title(
    "Monthly Average Rainfall Heatmap (2024–2025)",
    fontsize=18,
    fontweight="bold",
    pad=18
)

plt.xlabel("Month", fontsize=14, fontweight="bold")
plt.ylabel("Year", fontsize=14, fontweight="bold")

plt.xticks(fontsize=12, fontweight="bold")
plt.yticks(fontsize=12, fontweight="bold", rotation=0)

plt.tight_layout()

plt.savefig(
    f"{output_folder}/06_monthly_rainfall_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from pyspark.sql import functions as F

# Step 1: Categorize rainfall intensity
rainfall_category_df = valid_df.withColumn(
    "rainfall_category",
    F.when(F.col("rainfall_mm") == 0, "No Rain")
    .when(F.col("rainfall_mm") <= 2.5, "Light Rain")
    .when(F.col("rainfall_mm") <= 15, "Moderate Rain")
    .when(F.col("rainfall_mm") <= 64.5, "Heavy Rain")
    .otherwise("Very Heavy Rain")
)

# Step 2: Count observations in each category
category_counts = rainfall_category_df.groupBy(
    "rainfall_category"
).agg(
    F.count("*").alias("total_observations")
)

# Step 3: Convert to Pandas
category_pd = category_counts.toPandas()

# Step 4: Arrange categories in order
category_order = [
    "No Rain",
    "Light Rain",
    "Moderate Rain",
    "Heavy Rain",
    "Very Heavy Rain"
]

category_pd["rainfall_category"] = pd.Categorical(
    category_pd["rainfall_category"],
    categories=category_order,
    ordered=True
)

category_pd = category_pd.sort_values("rainfall_category")

# Step 5: Calculate percentages
category_pd["percentage"] = (
    category_pd["total_observations"] /
    category_pd["total_observations"].sum()
) * 100

# Step 6: Create graph
plt.figure(figsize=(12, 7))

colors = [
    "#1976D2",
    "#00BFA5",
    "#FFC107",
    "#FF7043",
    "#D50000"
]

bars = plt.bar(
    category_pd["rainfall_category"].astype(str),
    category_pd["total_observations"],
    color=colors,
    edgecolor="black",
    linewidth=1.2
)

# Add percentage labels
for bar, percentage in zip(
    bars,
    category_pd["percentage"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{percentage:.1f}%",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

# Titles and labels
plt.title(
    "Distribution of Rainfall Intensity Categories",
    fontsize=18,
    fontweight="bold",
    pad=20
)

plt.xlabel(
    "Rainfall Intensity",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "Number of Observations",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(
    fontsize=11,
    fontweight="bold",
    rotation=10
)

plt.yticks(fontsize=11, fontweight="bold")

plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()

# Save graph
plt.savefig(
    f"{output_folder}/06_rainfall_intensity_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Display category statistics
print("RAINFALL INTENSITY DISTRIBUTION")
print(category_pd.to_string(index=False))

In [ ]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Step 1: Classify daily rainfall intensity
extreme_df = valid_df.withColumn(
    "rainfall_intensity",
    F.when(
        (F.col("rainfall_mm") >= 64.5) &
        (F.col("rainfall_mm") <= 115.5),
        "Heavy Rain"
    )
    .when(
        (F.col("rainfall_mm") > 115.5) &
        (F.col("rainfall_mm") <= 204.4),
        "Very Heavy Rain"
    )
    .when(
        F.col("rainfall_mm") >= 204.5,
        "Extremely Heavy Rain"
    )
)

# Step 2: Keep only high-intensity rainfall observations
extreme_only = extreme_df.filter(
    F.col("rainfall_intensity").isNotNull()
)

# Step 3: Count observations by year and intensity
extreme_summary = extreme_only.groupBy(
    F.year("date").alias("year"),
    "rainfall_intensity"
).agg(
    F.count("*").alias("observation_count")
)

extreme_pd = extreme_summary.toPandas()

# Step 4: Prepare comparison table
category_order = [
    "Heavy Rain",
    "Very Heavy Rain",
    "Extremely Heavy Rain"
]

comparison = extreme_pd.pivot(
    index="rainfall_intensity",
    columns="year",
    values="observation_count"
).fillna(0)

comparison = comparison.reindex(category_order).fillna(0)

comparison = comparison.reindex(columns=[2024, 2025], fill_value=0)

print("EXTREME RAINFALL COMPARISON: 2024 vs 2025")
print(comparison)

# Step 5: Plot grouped bar chart
x = np.arange(len(category_order))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 7))

bars1 = ax.bar(
    x - width/2,
    comparison[2024],
    width,
    label="2024",
    color="#1565C0",
    edgecolor="black"
)

bars2 = ax.bar(
    x + width/2,
    comparison[2025],
    width,
    label="2025",
    color="#FF6D00",
    edgecolor="black"
)

# Add observation counts above bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f"{int(height):,}",
            (bar.get_x() + bar.get_width()/2, height),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            fontsize=10,
            fontweight="bold"
        )

ax.set_title(
    "Extreme Rainfall Comparison (2024 vs 2025)",
    fontsize=17,
    fontweight="bold",
    pad=18
)

ax.set_xlabel(
    "Rainfall Intensity Category",
    fontsize=13,
    fontweight="bold"
)

ax.set_ylabel(
    "Number of Grid-Cell Observations",
    fontsize=13,
    fontweight="bold"
)

ax.set_xticks(x)
ax.set_xticklabels(
    category_order,
    fontsize=11,
    fontweight="bold"
)

ax.legend(title="Year", fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{output_folder}/07_extreme_rainfall_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import os
import glob
import shutil
from google.colab import files

# Folder containing all generated graphs
output_folder = "/content/LAKE2026_Rainfall_Graphs"

# Find all PNG graph files
graph_files = glob.glob(os.path.join(output_folder, "*.png"))

# Check whether graphs exist
if graph_files:
    print("Graphs found:")
    for graph in sorted(graph_files):
        print(os.path.basename(graph))

    # Create ZIP file
    zip_path = "/content/LAKE2026_All_Rainfall_Graphs.zip"

    shutil.make_archive(
        "/content/LAKE2026_All_Rainfall_Graphs",
        "zip",
        output_folder
    )

    # Download ZIP
    files.download(zip_path)

    print(f"\nTotal graphs: {len(graph_files)}")
else:
    print("No graphs found. Please check your output folder.")

In [ ]:
# Install reverse-geocoding library
!pip install geopy -q

import time
import pandas as pd
import matplotlib.pyplot as plt
from geopy.geocoders import Nominatim

# Step 1: Find the top 10 rainfall grid locations
hotspots = valid_df.groupBy(
    "latitude", "longitude"
).agg(
    F.avg("rainfall_mm").alias("average_rainfall")
).orderBy(
    F.desc("average_rainfall")
).limit(10).toPandas()

# Step 2: Convert coordinates into nearby place names
geolocator = Nominatim(user_agent="lake2026_rainfall_research")

def get_place_name(latitude, longitude):
    try:
        location = geolocator.reverse(
            (latitude, longitude),
            exactly_one=True,
            language="en",
            zoom=10,
            addressdetails=True
        )

        if location:
            address = location.raw.get("address", {})

            place = (
                address.get("city") or
                address.get("town") or
                address.get("village") or
                address.get("municipality") or
                address.get("suburb") or
                address.get("county") or
                address.get("state_district")
            )

            district = (
                address.get("state_district") or
                address.get("county")
            )

            state = address.get("state")

            if place and district and place != district:
                return f"{place}, {district}"

            if place:
                return f"{place}, {state}" if state else place

            if district:
                return f"{district}, {state}" if state else district

            return state if state else "Place name unavailable"

        return "Place name unavailable"

    except Exception as e:
        print(f"Geocoding issue: {e}")
        return "Place name unavailable"


place_names = []

for _, row in hotspots.iterrows():
    name = get_place_name(row["latitude"], row["longitude"])
    place_names.append(name)
    time.sleep(1.1)  # Respect geocoding service limits

hotspots["place_name"] = place_names

# Step 3: Remove coordinates from the displayed results
hotspots = hotspots[
    ["place_name", "average_rainfall"]
]

# Step 4: Plot top rainfall hotspots
plt.figure(figsize=(13, 7))

bars = plt.barh(
    hotspots["place_name"],
    hotspots["average_rainfall"],
    color="#FF7043",
    edgecolor="black",
    linewidth=1.2
)

plt.gca().invert_yaxis()

for bar in bars:
    plt.text(
        bar.get_width(),
        bar.get_y() + bar.get_height()/2,
        f" {bar.get_width():.2f} mm",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

plt.title(
    "Top 10 Rainfall Hotspot Places (2024–2025)",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel(
    "Mean Observed Rainfall (mm)",
    fontsize=13,
    fontweight="bold"
)

plt.ylabel(
    "Place Name",
    fontsize=13,
    fontweight="bold"
)

plt.yticks(fontsize=10, fontweight="bold")
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()

plt.savefig(
    f"{output_folder}/09_rainfall_hotspots_by_place.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("TOP 10 RAINFALL HOTSPOTS")
print(hotspots.to_string(index=False))

In [ ]:
annual_max = valid_df.groupBy(
    F.year("date").alias("year")
).agg(
    F.max("rainfall_mm").alias("maximum_rainfall")
).orderBy("year").toPandas()

plt.figure(figsize=(9, 6))

# Vibrant purple and teal colours
colors = ["#8E24AA", "#00BFA5"]

bars = plt.bar(
    annual_max["year"].astype(str),
    annual_max["maximum_rainfall"],
    color=colors,
    edgecolor="black",
    linewidth=1.2,
    width=0.5
)

# Value labels
for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f'{bar.get_height():.2f} mm',
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

plt.title(
    "Annual Maximum Daily Grid-Cell Rainfall",
    fontsize=16,
    fontweight="bold",
    pad=15
)

plt.xlabel("Year", fontsize=13, fontweight="bold")
plt.ylabel("Maximum Rainfall (mm)", fontsize=13, fontweight="bold")

plt.xticks(fontsize=12, fontweight="bold")
plt.yticks(fontsize=11)

plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{output_folder}/11_annual_maximum_rainfall.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(annual_max.to_string(index=False))